In [1]:
from langgraph_sdk import get_client

In [2]:
alice = get_client(
    url="http://localhost:2024",
    headers={"Authorization": "Bearer user1-token"}
)

bob = get_client(
    url="http://localhost:2024",
    headers={"Authorization": "Bearer user2-token"}
)

In [ ]:


# Alice creates an assistant
alice_assistant = await alice.assistants.create(graph_id="agent")
print(f"Alice created assistant: {alice_assistant['assistant_id']}")

# Alice creates a thread
alice_thread = await alice.threads.create()
print(f"Alice created thread: {alice_thread['thread_id']}")

await alice.runs.create(
    thread_id=alice_thread["thread_id"],
    assistant_id="agent",
    input={"messages": [{"role": "user", "content": "Hello from Alice"}]}
)

# Bob tries to read Alice's thread
try:
    await bob.threads.get(alice_thread["thread_id"])
    print("Bob should not be able to see this!")
except Exception as e:
    print("Correctly denied access to Bob:", e)

# Bob creates his own thread
bob_thread = await bob.threads.create()
print(f"Bob created his own thread: {bob_thread['thread_id']}")

Alice created assistant: e8893060-7da8-4112-940f-248d779ffd42
Alice created thread: 3cfb72a8-e74e-4f07-b021-8b35f60d35a7
Correctly denied access to Bob: Client error '404 Not Found' for url 'http://localhost:2024/threads/3cfb72a8-e74e-4f07-b021-8b35f60d35a7'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404
Bob created his own thread: b45f9d6e-478b-4144-ab7e-e5641c13350d


Add the following handler to `src/security/authon.py`

```python
@auth.on.threads.create
async def on_thread_create(ctx: Auth.types.AuthContext, value: dict):
    metadata = value.setdefault("metadata", {})
    metadata["owner"] = ctx.user.identity
    return {"owner": ctx.user.identity}

@auth.on.threads.read
async def on_thread_read(ctx: Auth.types.AuthContext, value: dict):
    return {"owner": ctx.user.identity}

@auth.on.assistants
async def on_assistants(ctx: Auth.types.AuthContext, value: dict):
  if ctx.user.identity == "user2":
    raise Auth.exceptions.HTTPException(
        status_code=403,
        detail="User lacks the required permissions."
    )
  else:
    return {
        "identity": ctx.user.identity,
        "is_authenticated": True
    }
```

After adding and saving this, run the following cell

In [ ]:
from langgraph_sdk import get_client

alice = get_client(
    url="http://localhost:2024",
    headers={"Authorization": "Bearer user1-token"}
)

bob = get_client(
    url="http://localhost:2024",
    headers={"Authorization": "Bearer user2-token"}
)

In [3]:
try:
    await alice.assistants.create("agent")
    print("Alice should be able to create assistants!")
except Exception as e:
    print("Incorrectly denied assistant creation for Alice:", e)

try:
    await bob.assistants.create("agent")
    print("bob should not be able to create assistants!")
except Exception as e:
    print("Correctly denied assistant creation for Bob:", e)

Alice should be able to create assistants!
Correctly denied assistant creation for Bob: Client error '403 Forbidden' for url 'http://localhost:2024/assistants'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403


In [4]:
# Check thread creation still works
alice_thread = await alice.threads.create()
print(f"Alice created thread: {alice_thread['thread_id']}")

bob_thread = await bob.threads.create()
print(f"Bob created thread: {bob_thread['thread_id']}")

Alice created thread: 4526f0d3-383c-4307-93d7-2f32e4650a48
Bob created thread: 3a656737-7d5f-46fe-8b5f-02a74ee80ee9
